# TC Reduction with Bayesian Optimization — Firebase Chat

Eksperimen ini menguji apakah BO dapat mereduksi subset TC QA tanpa kehilangan terlalu banyak bug recall.
Stopping criterion menghentikan BO sebelum semua 69 TC dijalankan.

**Status:** Eksperimen aktif (alternatif interpretasi arahan sensei)
**Referensi:** BO_2_TC_REDUCTION_ALTERNATIF.md, BO_3_KONTEKS_UTAMA_DISKUSI.md

In [ ]:
import os

# === EXPERIMENT CONFIGURATION ===
# Vectorizer (sama dengan prioritization)
PAPER_VECTORISERS = ("TF-IDF", "Feature Hashing", "Word2Vec", "GloVe",
                     "FastText", "ELMo", "Flair")
ADDITIONAL_VECTORISERS = ("One-Hot", "E5-Large")
EXPERIMENT_METHODS = PAPER_VECTORISERS + ADDITIONAL_VECTORISERS

# Kernel — EXP2 (env var: BO_KERNEL)
# Options: cosine, rbf, matern32, matern52, rq
KERNEL_NAME = os.getenv("BO_KERNEL", "cosine")

# HP Tuning — EXP3 (env var: BO_HP_TUNING)
# Options: fixed, mle
HP_TUNING = os.getenv("BO_HP_TUNING", "fixed")

# Acquisition Function — EXP4 (env var: BO_AF)
# Options: cost_ucb, ei, logei, ucb, pi, ts
AF_NAME = os.getenv("BO_AF", "cost_ucb")

# === REDUCTION-SPECIFIC PARAMETERS ===
# Stopping threshold theta — EXP6
# BO berhenti jika semua kandidat tersisa punya upper bound (mean + beta*std) < STOP_THETA
STOP_THETA = float(os.getenv("BO_STOP_THETA", "0.1"))

# Budget constraint — EXP7
# Maksimum TC yang boleh dipilih BO (dari 69 total)
MAX_TC = int(os.getenv("BO_MAX_TC", "35"))

# GP hyperparameters (fixed default)
SIGMA2 = float(os.getenv("BO_SIGMA2", "1.0"))
NOISE_ALPHA = float(os.getenv("BO_NOISE_ALPHA", "0.01"))
UCB_BETA = float(os.getenv("BO_UCB_BETA", "2.0"))

# Fairness replicates
N_FAIRNESS_REPLICATES = int(os.getenv("BO_FAIRNESS_REPS", "10"))

print(f"Config: kernel={KERNEL_NAME}, af={AF_NAME}, hp={HP_TUNING}")
print(f"Reduction: stop_theta={STOP_THETA}, max_tc={MAX_TC}")

In [ ]:
import importlib.util
import os
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Always needed
for pkg in ["openpyxl", "scikit-learn", "scipy", "matplotlib", "seaborn", "pandas", "numpy"]:
    if importlib.util.find_spec(pkg.split("[")[0].replace("-","_")) is None:
        install(pkg)

VECTORISER_MODE = os.getenv("HAKUSAN_VECTORISER_MODE", "quick")
if VECTORISER_MODE == "all":
    for pkg in ["gensim", "tensorflow", "tensorflow-hub", "flair>=0.14,<0.15",
                "sentence-transformers>=2.7,<3.0", "setuptools<70"]:
        install(pkg)
    print("All vectoriser packages installed.")
else:
    print("Quick mode: only TF-IDF and Feature Hashing available.")

In [ ]:
from pathlib import Path
import hashlib, json, math, re, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import ndtr
from scipy.optimize import minimize
warnings.filterwarnings("ignore")

## 1. Load QA workbook (69 TC Firebase Chat)

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

def parse_minutes(value):
    if pd.isna(value): return 5.0
    s = str(value).strip().lower()
    m = re.match(r"(\d+(?:\.\d+)?)\s*(?:menit|min|m)?", s)
    return float(m.group(1)) if m else 5.0

def find_workbook():
    curr = Path(".").resolve()
    search_roots = [curr] + list(curr.parents) + [
        Path(r"C:/Users/radit/Project/VisualStudioProject/Skripsi/MAS AI").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Suitmedia").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Kepake").resolve()
    ]
    for root in search_roots:
        if root.exists():
            target = root / "scenarios" / "firebase_chat" / "scenario.xlsx"
            if target.exists():
                return target
            for p in [root] + list(root.parents):
                found = list(p.glob("**/scenarios/firebase_chat/scenario.xlsx")) + list(p.glob("**/scenario.xlsx"))
                if found:
                    return found[0]
    raise FileNotFoundError("No scenario.xlsx found in standard paths.")

def load_qa_cases(path):
    df_raw = pd.read_excel(path, sheet_name=0, header=None)
    header_idx = None
    for idx, row in df_raw.iterrows():
        if any("TCS ID" in str(val) for val in row.values):
            header_idx = idx
            break
    if header_idx is not None:
        df = pd.read_excel(path, sheet_name=0, header=header_idx)
    else:
        df = df_raw
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(subset=["TCS ID"]).reset_index(drop=True)
    df = df[df["TCS ID"].astype(str).str.startswith("FC-")].reset_index(drop=True)
    df["cost_minutes"] = df.get("Time Testing", pd.Series([5.0]*len(df))).apply(parse_minutes)
    return df

WORKBOOK_PATH = find_workbook()
cases = load_qa_cases(WORKBOOK_PATH)
assert len(cases) == 69, f"Expected 69 cases, got {len(cases)}"
print(f"Loaded {len(cases)} test cases from {WORKBOOK_PATH.name}")
print(f"Total estimated time: {cases['cost_minutes'].sum():.0f} minutes")


In [ ]:
# Fixed dummy bug indices (13 dummy bugs out of 69 TC)
DUMMY_BUG_INDICES = {1, 4, 9, 13, 18, 22, 26, 30, 35, 40, 50, 57, 62}

oracle = np.array([1.0 if i in DUMMY_BUG_INDICES else 0.0 for i in range(len(cases))], dtype=float)

# Warm start: 1 TC per Menu (coverage-first)
menus = cases["Menu"].tolist()
seen_menus = set()
initial_indices = []
for i, menu in enumerate(menus):
    if menu not in seen_menus:
        seen_menus.add(menu)
        initial_indices.append(i)

print(f"Oracle: {int(oracle.sum())} dummy bugs in {len(cases)} TC")
print(f"Warm start: {len(initial_indices)} TC (1 per menu)")
print(f"Budget: max {MAX_TC} TC, stopping theta={STOP_THETA}")


## 2. Vectoriser matrices (same as prioritization EXP1)

In [ ]:
TEXT_FIELDS = [
    "Menu", "Submenu 1", "Submenu 2", "Test Case Scenario", "Test Step",
    "Expected Result"
]
available_cols = [c for c in TEXT_FIELDS if c in cases.columns]
corpus = cases[available_cols].fillna("").apply(lambda r: " ".join(r.astype(str)), axis=1).tolist()

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.preprocessing import normalize

def build_matrices(methods, corpus):
    matrices = {}
    for method in methods:
        if method == "TF-IDF":
            vec = TfidfVectorizer(max_features=500)
            X = vec.fit_transform(corpus).toarray()
        elif method == "Feature Hashing":
            vec = HashingVectorizer(n_features=256, alternate_sign=False)
            X = vec.transform(corpus).toarray()
        elif method == "One-Hot":
            tokens = [set(doc.lower().split()) for doc in corpus]
            vocab = sorted(set(w for t in tokens for w in t))
            X = np.array([[1.0 if w in t else 0.0 for w in vocab] for t in tokens])
        else:
            # Placeholder for GPU vectorisers
            X = np.random.RandomState(42).randn(len(corpus), 64)
        X = normalize(X, norm="l2")
        matrices[method] = X
        print(f"  {method}: shape {X.shape}")
    return matrices

active_methods = ["TF-IDF", "Feature Hashing"] if os.getenv("HAKUSAN_VECTORISER_MODE","quick") == "quick" else list(EXPERIMENT_METHODS)
print(f"Building matrices for: {active_methods}")
matrices = build_matrices(active_methods, corpus)

## 3. GPC Core + Reduction Loop

Perbedaan dari prioritization:
- `run_reduction_loop()` menghentikan BO ketika stopping criterion terpenuhi
- Stopping criterion: semua kandidat tersisa punya `mean + beta*std < STOP_THETA` ATAU sudah pilih `MAX_TC` TC
- Output tambahan: `bug_recall` = bugs_found / total_bugs pada saat berhenti

In [ ]:
import numpy as np
from scipy.special import ndtr
from scipy.optimize import minimize as sp_minimize

# ── Kernel functions ─────────────────────────────────────────────────────────────

def _cosine_kernel(X, Y):
    return X @ Y.T

def _rbf_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    return np.exp(-0.5 * np.sum(diff**2, axis=-1) / length_scale**2)

def _matern32_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(3)*v) * np.exp(-np.sqrt(3)*v)

def _matern52_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(5)*v + 5*v**2/3) * np.exp(-np.sqrt(5)*v)

def _rq_kernel(X, Y, length_scale=1.0, alpha=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r2 = np.sum(diff**2, axis=-1)
    return (1 + r2 / (2*alpha*length_scale**2))**(-alpha)

def get_kernel(name, X, Y, sigma2=1.0, length_scale=1.0):
    if name == "cosine":
        return sigma2 * _cosine_kernel(X, Y)
    elif name == "rbf":
        return sigma2 * _rbf_kernel(X, Y, length_scale)
    elif name == "matern32":
        return sigma2 * _matern32_kernel(X, Y, length_scale)
    elif name == "matern52":
        return sigma2 * _matern52_kernel(X, Y, length_scale)
    elif name == "rq":
        return sigma2 * _rq_kernel(X, Y, length_scale)
    else:
        raise ValueError(f"Unknown kernel: {name}")

# ── Probit derivatives (Laplace GPC) ────────────────────────────────────────────

def _probit_derivatives(f, y):
    yf = y * f
    phi = ndtr(yf)
    phi = np.clip(phi, 1e-10, 1 - 1e-10)
    pdf = np.exp(-0.5 * yf**2) / np.sqrt(2 * np.pi)
    grad = y * pdf / phi
    W = (pdf / phi)**2 + yf * pdf / phi
    return grad, W

# ── MLE for kernel hyperparameters (EXP3) ───────────────────────────────────────

def _neg_lml(log_params, X, y, kernel_name, noise):
    sigma2 = np.exp(log_params[0])
    length_scale = np.exp(log_params[1]) if len(log_params) > 1 else 1.0
    n = len(y)
    K = get_kernel(kernel_name, X, X, sigma2, length_scale)
    K += noise * np.eye(n)
    try:
        L = np.linalg.cholesky(K)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
        lml = -0.5 * y @ alpha - np.sum(np.log(np.diag(L))) - 0.5 * n * np.log(2*np.pi)
        return -lml
    except np.linalg.LinAlgError:
        return 1e10

def tune_hyperparams(X, y, kernel_name, noise, n_restarts=5):
    best_val, best_params = np.inf, [0.0, 0.0]
    for _ in range(n_restarts):
        x0 = np.random.randn(2)
        res = sp_minimize(_neg_lml, x0, args=(X, y, kernel_name, noise),
                          method="L-BFGS-B")
        if res.fun < best_val:
            best_val = res.fun
            best_params = res.x
    return np.exp(best_params[0]), np.exp(best_params[1])

# ── GPC predict (Laplace approximation) ──────────────────────────────────────────

def _gp_predict(train_x, train_y, cand_x,
                sigma2=SIGMA2, noise=NOISE_ALPHA,
                kernel_name=KERNEL_NAME, hp_tuning=HP_TUNING):
    if hp_tuning == "mle" and len(train_y) >= 5:
        sigma2, length_scale = tune_hyperparams(train_x, train_y*2-1, kernel_name, noise)
    else:
        length_scale = 1.0

    train_y_pm = train_y * 2 - 1  # {0,1} -> {-1,+1}
    n = len(train_y)
    K = get_kernel(kernel_name, train_x, train_x, sigma2, length_scale)
    K += noise * np.eye(n)

    # Laplace: Newton iterations
    f = np.zeros(n)
    for _ in range(20):
        grad, W = _probit_derivatives(f, train_y_pm)
        W_safe = np.maximum(W, 1e-8)
        W_sqrt = np.sqrt(W_safe)
        B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
        L = np.linalg.cholesky(B + 1e-8 * np.eye(n))
        b = W_safe * f + grad
        v = np.linalg.solve(L, W_sqrt * (K @ b))
        f_new = K @ (b - W_sqrt * np.linalg.solve(L.T, v))
        if np.max(np.abs(f_new - f)) < 1e-6:
            f = f_new
            break
        f = f_new

    grad, W = _probit_derivatives(f, train_y_pm)
    W_safe = np.maximum(W, 1e-8)
    W_sqrt = np.sqrt(W_safe)
    B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
    L = np.linalg.cholesky(B + 1e-8 * np.eye(n))

    K_star = get_kernel(kernel_name, cand_x, train_x, sigma2, length_scale)
    mu = K_star @ grad
    v = np.linalg.solve(L, W_sqrt[:, None] * K_star.T)
    K_ss = np.diag(get_kernel(kernel_name, cand_x, cand_x, sigma2, length_scale))
    var = np.maximum(K_ss - np.sum(v**2, axis=0), 1e-8)
    std = np.sqrt(var)

    # Squash to [0,1] probability
    kappa = 1.0 / np.sqrt(1 + np.pi * var / 8)
    mean_prob = ndtr(kappa * mu)
    return mean_prob, std

# ── Acquisition functions ─────────────────────────────────────────────────────────

def _cost_ucb(mean, std, cost, beta=UCB_BETA):
    return (mean + beta * std) / np.sqrt(np.maximum(cost, 1.0))

def _ei(mean, std, best, xi=0.01):
    z = (mean - best - xi) / np.maximum(std, 1e-8)
    return (mean - best - xi) * ndtr(z) + std * np.exp(-0.5*z**2)/np.sqrt(2*np.pi)

def _logei(mean, std, best, xi=0.01):
    ei = _ei(mean, std, best, xi)
    return np.log(np.maximum(ei, 1e-30))

def _ucb(mean, std, beta=UCB_BETA):
    return mean + beta * std

def _pi(mean, std, best, xi=0.01):
    return ndtr((mean - best - xi) / np.maximum(std, 1e-8))

def _thompson(mean, std, rng):
    return rng.normal(mean, std)

def get_af_score(af_name, mean, std, cost, best, rng):
    if af_name == "cost_ucb":
        return _cost_ucb(mean, std, cost)
    elif af_name == "ei":
        return _ei(mean, std, best)
    elif af_name == "logei":
        return _logei(mean, std, best)
    elif af_name == "ucb":
        return _ucb(mean, std)
    elif af_name == "pi":
        return _pi(mean, std, best)
    elif af_name == "ts":
        return _thompson(mean, std, rng)
    else:
        raise ValueError(f"Unknown AF: {af_name}")

# ── Stopping criterion ─────────────────────────────────────────────────────────────

def _should_stop(mean, std, beta, theta):
    """Return True jika semua kandidat tersisa punya upper bound < theta."""
    upper_bounds = mean + beta * std
    return bool(np.all(upper_bounds < theta))

# ── Reduction loop (CORE PERBEDAAN dari prioritization) ────────────────────────────

def run_reduction_loop(matrix, ids, menus, costs, oracle,
                       initial_indices,
                       max_tc=MAX_TC,
                       stop_theta=STOP_THETA,
                       kernel_name=KERNEL_NAME,
                       af_name=AF_NAME,
                       hp_tuning=HP_TUNING,
                       sigma2=SIGMA2,
                       noise=NOISE_ALPHA,
                       beta=UCB_BETA,
                       rng=None):
    if rng is None:
        rng = np.random.default_rng(42)

    n = len(ids)
    selected = list(initial_indices)
    observed_oracle = {i: oracle[i] for i in selected}
    total_bugs = int(oracle.sum())

    history = []  # list of dicts per iteration

    for step in range(max_tc - len(initial_indices)):
        candidates = [i for i in range(n) if i not in set(selected)]
        if not candidates:
            break

        train_x = matrix[selected]
        train_y = np.array([observed_oracle[i] for i in selected])
        cand_x = matrix[candidates]
        cand_costs = np.array([costs[i] for i in candidates])

        mean, std = _gp_predict(train_x, train_y, cand_x,
                                sigma2=sigma2, noise=noise,
                                kernel_name=kernel_name, hp_tuning=hp_tuning)

        # Stopping check
        if _should_stop(mean, std, beta, stop_theta):
            break

        # AF selection
        best_so_far = float(np.mean(train_y)) if train_y.sum() > 0 else 0.0
        scores = get_af_score(af_name, mean, std, cand_costs, best_so_far, rng)
        chosen_local = int(np.argmax(scores))
        chosen = candidates[chosen_local]

        selected.append(chosen)
        observed_oracle[chosen] = oracle[chosen]

        bugs_found = sum(observed_oracle[i] for i in selected)
        recall = bugs_found / total_bugs if total_bugs > 0 else 0.0
        total_cost = sum(costs[i] for i in selected)

        history.append({
            "step": len(selected),
            "chosen_id": ids[chosen],
            "bugs_found": bugs_found,
            "bug_recall": recall,
            "total_cost_min": total_cost,
            "stopped_early": False
        })

    # Mark if stopped by theta (not by max_tc)
    stopped_by_theta = len(selected) < max_tc and len(selected) < n
    bugs_found = sum(observed_oracle[i] for i in selected)
    recall = bugs_found / total_bugs if total_bugs > 0 else 0.0

    return {
        "selected_indices": selected,
        "n_selected": len(selected),
        "bugs_found": bugs_found,
        "bug_recall": recall,
        "stopped_by_theta": stopped_by_theta,
        "total_cost_min": sum(costs[i] for i in selected),
        "history": history
    }

print("GPC + Reduction loop defined.")
print(f"  kernel={KERNEL_NAME}, af={AF_NAME}, hp={HP_TUNING}")
print(f"  stop_theta={STOP_THETA}, max_tc={MAX_TC}")

## 4. EXP1 — Vectorizer comparison (reduction scenario)

Jalankan reduction loop untuk setiap vectorizer.
Metric: bug_recall saat berhenti, n_selected (berapa TC yang dijalankan).

In [ ]:
costs = cases["cost_minutes"].values
ids = cases["TCS ID"].tolist()

reduction_results = {}
for method_name, matrix in matrices.items():
    result = run_reduction_loop(
        matrix=matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        max_tc=MAX_TC,
        stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        hp_tuning=HP_TUNING,
        rng=np.random.default_rng(42)
    )
    reduction_results[method_name] = result
    print(f"{method_name:20s}: ran {result['n_selected']:2d} TC, "
          f"recall={result['bug_recall']:.1%}, "
          f"cost={result['total_cost_min']:.0f} min, "
          f"stopped_by_theta={result['stopped_by_theta']}")

## 5. Summary — Reduction results per vectorizer

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json, zipfile

os.makedirs("results", exist_ok=True)

summary_rows = []
for method, r in reduction_results.items():
    summary_rows.append({
        "Method": method,
        "TC Dijalankan": r["n_selected"],
        "TC Dijalankan (%)": f"{r['n_selected']/69*100:.1f}%",
        "Bug Recall": f"{r['bug_recall']:.1%}",
        "Bugs Found": r["bugs_found"],
        "Cost (menit)": f"{r['total_cost_min']:.0f}",
        "Stopped by theta": r["stopped_by_theta"]
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("results/exp1_vectorizer_summary.csv", index=False)
with open('results/exp1_reduction_results.json', 'w') as f:
    json.dump({k: {rk: (rv.tolist() if isinstance(rv, np.ndarray) else bool(rv) if isinstance(rv, (bool, np.bool_)) else int(rv) if isinstance(rv, (int, np.integer)) else float(rv) if isinstance(rv, (float, np.floating)) else rv) for rk, rv in v.items()} for k, v in reduction_results.items()}, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 5))
methods = list(reduction_results.keys())
recalls = [reduction_results[m]["bug_recall"] for m in methods]
n_selected = [reduction_results[m]["n_selected"] for m in methods]
colors = sns.color_palette("tab10", len(methods))
bars = ax.bar(methods, n_selected, color=colors)
ax2 = ax.twinx()
ax2.plot(methods, recalls, "D--k", label="Bug Recall")
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Bug Recall")
ax.set_ylabel("TC Dijalankan")
ax.set_title(f"Reduction: TC Dijalankan vs Bug Recall\n(theta={STOP_THETA}, max_tc={MAX_TC})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("results/reduction_summary.png", dpi=100)
plt.show()
print("EXP1 CSV & JSON saved to results/")


## 6. EXP6 — Stopping threshold sweep

Sweep theta ∈ {0.05, 0.1, 0.2, 0.3} untuk vectorizer terbaik dari EXP1.
Lihat trade-off: theta kecil = lebih banyak TC dijalankan, recall lebih tinggi.
theta besar = lebih sedikit TC, recall lebih rendah.

In [ ]:
THETA_VALUES = [0.05, 0.1, 0.2, 0.3]

best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
print(f"EXP6: theta sweep menggunakan vectorizer '{best_method}'")

theta_results = []
for theta in THETA_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        max_tc=MAX_TC,
        stop_theta=theta,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        rng=np.random.default_rng(42)
    )
    theta_results.append({
        "theta": theta,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  theta={theta}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_theta = pd.DataFrame(theta_results)
df_theta.to_csv("results/exp6_theta_sweep.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(df_theta["theta"], df_theta["TC_run"], "o-b", label="TC Dijalankan")
ax1.set_xlabel("Stopping Threshold (theta)")
ax1.set_ylabel("TC Dijalankan", color="b")
ax2 = ax1.twinx()
ax2.plot(df_theta["theta"], df_theta["bug_recall"], "s--r", label="Bug Recall")
ax2.set_ylabel("Bug Recall", color="r")
ax2.set_ylim(0, 1.1)
ax1.set_title(f"EXP6: Theta Sweep (vectorizer={best_method})")
plt.tight_layout()
plt.savefig("results/exp6_theta_sweep.png", dpi=100)
plt.show()
print(df_theta.to_string(index=False))


## 7. EXP7 — Budget constraint sweep

Sweep MAX_TC ∈ {20, 30, 40, 50} dari 69 total TC.
Lihat berapa bug recall yang bisa dicapai dengan budget terbatas.

In [ ]:
BUDGET_VALUES = [20, 30, 40, 50]

print(f"EXP7: budget sweep menggunakan vectorizer '{best_method}'")

budget_results = []
for max_tc in BUDGET_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        max_tc=max_tc,
        stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        rng=np.random.default_rng(42)
    )
    budget_results.append({
        "max_tc": max_tc,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"]
    })
    print(f"  max_tc={max_tc}: ran {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_budget = pd.DataFrame(budget_results)
df_budget.to_csv("results/exp7_budget_sweep.csv", index=False)

random_recalls = []
rng_base = np.random.default_rng(0)
for max_tc in BUDGET_VALUES:
    recalls_seed = []
    for seed in range(20):
        rng_s = np.random.default_rng(seed)
        idx = rng_s.choice(69, size=max_tc, replace=False)
        recalls_seed.append(oracle[idx].sum() / oracle.sum())
    random_recalls.append(np.mean(recalls_seed))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_budget["max_tc"], df_budget["bug_recall"], "o-b", label="BO Reduction")
ax.plot(BUDGET_VALUES, random_recalls, "s--r", label="Random Baseline")
ax.axhline(1.0, color="gray", linestyle=":", label="Full suite (69 TC)")
ax.set_xlabel("Budget (max TC)")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP7: Budget Sweep (theta={STOP_THETA})")
ax.legend()
plt.tight_layout()
plt.savefig("results/exp7_budget_sweep.png", dpi=100)
plt.show()
print(df_budget.to_string(index=False))


## 8. Fairness check (synthetic oracle)

Pastikan hasil tidak bergantung pada posisi dummy bug yang dipilih.
Jalankan dengan oracle acak (N_FAIRNESS_REPLICATES kali).

In [ ]:
def draw_uniform_oracle(case_count, bug_count, random_state):
    rng_f = np.random.default_rng(random_state)
    oracle_f = np.zeros(case_count)
    idx = rng_f.choice(case_count, size=bug_count, replace=False)
    oracle_f[idx] = 1.0
    return oracle_f

fairness_rows = []
for method_name, matrix in matrices.items():
    for rep in range(N_FAIRNESS_REPLICATES):
        oracle_f = draw_uniform_oracle(69, 13, rep)
        r = run_reduction_loop(
            matrix=matrix,
            ids=ids,
            menus=menus,
            costs=costs,
            oracle=oracle_f,
            initial_indices=initial_indices,
            max_tc=MAX_TC,
            stop_theta=STOP_THETA,
            kernel_name=KERNEL_NAME,
            af_name=AF_NAME,
            rng=np.random.default_rng(rep)
        )
        fairness_rows.append({
            "method": method_name,
            "rep": rep,
            "n_selected": r["n_selected"],
            "bug_recall": r["bug_recall"]
        })

df_fairness = pd.DataFrame(fairness_rows)
df_fair_summary = df_fairness.groupby("method").agg(
    mean_recall=("bug_recall", "mean"),
    std_recall=("bug_recall", "std"),
    mean_tc=("n_selected", "mean")
).round(3)
print("Fairness summary (synthetic oracle):")
print(df_fair_summary.to_string())

os.makedirs("results", exist_ok=True)
df_fairness.to_csv("results/fairness_reduction.csv", index=False)
df_fair_summary.to_csv("results/fairness_reduction_summary.csv")
print("Saved to results/")

## 9. Cara membaca hasil

**EXP1 (vectorizer):** Bandingkan `bug_recall` dan `TC_dijalankan` antar vectorizer.
Vectorizer terbaik = recall tinggi dengan TC sedikit.

**EXP6 (theta sweep):** Trade-off theta vs recall. 
- theta kecil (0.05) = BO lebih konservatif = lebih banyak TC = recall lebih tinggi
- theta besar (0.3) = BO lebih agresif berhenti = lebih sedikit TC = recall lebih rendah

**EXP7 (budget sweep):** Berapa minimal TC untuk recall yang masih acceptable?
Bandingkan dengan random baseline — kalau BO tidak mengalahkan random, ada masalah.

**Fairness check:** Jika semua metode punya recall ±0.05 di synthetic oracle, 
hasil EXP1 bisa dipercaya (bukan artefak posisi dummy bug).

**Batas interpretasi:**
- Dummy oracle ≠ bug nyata Firebase Chat
- Recall 100% di dummy tidak menjamin 100% di data nyata
- Untuk klaim reduction yang valid: butuh ground truth dari eksekusi nyata 69 TC

## 10. Package All Log Data & Visualizations into ZIP


In [ ]:
import zipfile
from pathlib import Path

results_dir = Path("results")
zip_path = Path("reduction_experiment_results.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in results_dir.glob('*'):
        if file.is_file():
            zipf.write(file, arcname=file.name)
            print(f"Zipped: {file.name}")

print(f"\nAll log data, CSVs, JSONs, and PNGs packaged into: {zip_path.resolve()}")
